# 06 | RAG主流文档加载器速查（PDF、Text、Markdown、目录）

CSV 和 JSON 已经单独讲过了，主要是让人更深的感受一下它的用法和适用场景。至于剩下的主流资料源，它们本质上都在做同一件事：把不同来源的资料加载成 LangChain 的 `Document`。

所以这篇只保留最常用、最容易在本地实际跑起来的几类：PDF、TXT、Markdown、文件夹批量加载，梭哈讲一下。

## 一、先看怎么选

| 资料类型 | 推荐 loader | 适合场景 |
| --- | --- | --- |
| PDF | `PyPDFLoader` | 法规、合同、制度、说明书 |
| TXT | `TextLoader` | 纯文本说明、日志摘录、简单文档 |
| Markdown | `UnstructuredMarkdownLoader` | 技术文档、知识库、README |
| 文件夹 | `DirectoryLoader` | 一次加载一批文件 |

不用背。先看你的资料长什么样，再选 loader。别为了用高级 loader，把简单事情搞复杂。

## 二、PDF：`PyPDFLoader`

适合加载法规、合同、制度、说明书这类 PDF。

默认建议先按页加载，因为页码对 RAG 很有用。用户问制度或法律问题时，最好能追到“第几页”。

In [ ]:
# 安装：uv add langchain-community pypdf
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("data/中华人民共和国数据安全法.pdf")

# mode="page"：一页 PDF 生成一个 Document。
# 好处是 metadata 里会保留 page / page_label，方便后续引用页码。
loader = PyPDFLoader(str(pdf_path), mode="page")
documents = loader.load()

print(len(documents))
print(documents[0].page_content[:500])
print(documents[0].metadata)

PDF 这里最该关注的是 `metadata`：

```text
source：PDF 文件路径
page：页码，从 0 开始
page_label：页面标签，通常更接近人看到的页码
total_pages：总页数
```

如果是扫描版 PDF，抽不出文字，就不要硬怪 `PyPDFLoader`。那通常需要 OCR。

## 三、纯文本：`TextLoader`

适合 `.txt`、简单日志、纯文本说明。

它很朴素：**一个文本文件默认读成一个 `Document`**。

所以就算 `xiaoshuo.txt` 很长，`len(documents)` 也通常是 `1`。这不是异常，说明 loader 已经把整份文本搬进来了。后面要不要按章节、段落、长度切开，是 `TextSplitter` 的事情。

In [ ]:
# 安装：uv add langchain-community
from pathlib import Path
from langchain_community.document_loaders import TextLoader

text_path = Path("data/xiaoshuo.txt")

# encoding 很重要。中文文本建议显式写 utf-8，少给自己挖坑。
loader = TextLoader(str(text_path), encoding="utf-8")
documents = loader.load()
print(len(documents))
print(documents[0].page_content[:500])
print(documents[0].metadata)

TXT 最大的问题不是加载，而是后续切分。

如果一个 TXT 很长，`TextLoader` 只是先读进来，后面通常还要接 `TextSplitter`。Loader 负责搬进门，切不切块是下一步。

## 四、Markdown：`UnstructuredMarkdownLoader`

适合技术文档、README、内部知识库、Notion 导出的 Markdown。

Markdown 比 TXT 多了标题、列表、链接这些结构。用普通 `TextLoader` 也能读，但 `UnstructuredMarkdownLoader` 更适合保留一些文档结构信息。

注意：这个 loader 依赖 `markdown` 包。如果没装，运行时可能会报 `ModuleNotFoundError: No module named 'markdown'`。

In [ ]:
# 安装：uv add langchain-community unstructured markdown
from pathlib import Path
from langchain_community.document_loaders import UnstructuredMarkdownLoader

md_path = Path("data/internal_refund_guide.md")
md_path.write_text(
    "# 退款操作手册\n\n"
    "## 未发货订单\n"
    "用户可以直接在订单详情页申请退款。\n\n"
    "## 已发货订单\n"
    "需要先确认物流状态，再进入售后流程。\n",
    encoding="utf-8",
)

# mode="single"：整篇 Markdown 作为一个 Document。
# mode="elements"：按标题、正文等元素拆成多个 Document，适合更细粒度处理。
loader = UnstructuredMarkdownLoader(str(md_path), mode="single")
documents = loader.load()

print(documents[0].page_content[:500])
print(documents[0].metadata)

Markdown 的选择很简单：

- 只是先加载进来：`mode="single"`
- 想按标题、段落等结构拆得更细：`mode="elements"`

别一上来就 elements。先看你的文档是不是值得这么细。

## 五、文件夹批量加载：`DirectoryLoader`

真实项目里，一个目录里往往不止一种文件。

比如：

```text
policies/
  refund.txt
  invoice.txt
  中华人民共和国数据安全法.pdf
```

这时候不要指望 `DirectoryLoader` 自动识别所有格式。它的作用是**批量找文件，然后把文件交给指定 loader**。

所以更清楚的做法是：按文件类型分批加载。

In [ ]:
# 安装：uv add langchain-community pypdf
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

folder = Path("data/policies")
folder.mkdir(exist_ok=True)

# 第一批：加载 txt 文件。
# DirectoryLoader 负责找到所有 txt；TextLoader 负责读取每个 txt。
txt_loader = DirectoryLoader(
    str(folder),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

# 第二批：加载 pdf 文件。
# DirectoryLoader 负责找到所有 pdf；PyPDFLoader 负责按页读取每个 pdf。
pdf_loader = DirectoryLoader(
    str(folder),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    loader_kwargs={"mode": "page"},
)

txt_documents = txt_loader.load()
pdf_documents = pdf_loader.load()

# 最后把不同类型加载出来的 Document 合并。
documents = txt_documents + pdf_documents

print("txt docs:", len(txt_documents))
print("pdf docs:", len(pdf_documents))
print("all docs:", len(documents))
print(documents[0].page_content[:300])
print(documents[0].metadata)

这个例子的重点是：

```text
DirectoryLoader 负责批量找文件
具体怎么读，仍然交给 TextLoader / PyPDFLoader 这类 loader
```

如果目录里文件类型很杂，推荐按后缀分批加载，再把结果合并。

这样比“一个 loader 试图包打天下”更清楚，也更容易排查问题。

## 六、最后怎么记

这篇保留四类就够了：

```text
PDF：PyPDFLoader，重点看页码 metadata
TXT：TextLoader，一个文件默认一个 Document
Markdown：UnstructuredMarkdownLoader，适合带标题结构的文档
文件夹：DirectoryLoader，批量找文件，再交给对应 loader 读取
```

它们最后都会变成 `Document`。

每次加载完，先看两件事：

```text
page_content 是不是我要的正文？
metadata 里有没有来源信息？
```

这两件事对了，文档加载这一步基本就稳了。
